In [43]:
import os
import sys
import urllib.request

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from stellargraph.layer import GCN_LSTM

In [2]:
parser = (lambda x:datetime.datetime.strptime(x, '%Y.%m.%d')) 
df = pd.read_csv('sp_beaches_update.csv', parse_dates=['Date'])
df = df.sort_values(by=['Date'])
df=df.loc[~df['Enterococcus'].isnull()]
#remover a praia do Leste, da cidade de iguape, pois esta praia sumiu por erosão em 2012
#remover a Lagoa Prumirim, da cidade de Ubatuba, pois esta praia possui somente 3 medições
df = df.loc[df['Beach']!='DO LESTE'].loc[df['Beach']!='LAGOA PRUMIRIM']
df['praia'] = df[['City', 'Beach']].apply(lambda x: ' - '.join(x), axis=1)
print(f'Numero de praias: {len(df.praia.unique())}')
df = df[['Date','praia','Enterococcus']]
df.shape

Numero de praias: 174


(69016, 3)

In [3]:
for praia in df.praia.unique():
    df_aux = df.loc[df['praia']==praia]
    print(f'Praia: {praia} - Qtd Medições: {len(df_aux.Date)}')

Praia: BERTIOGA - BORACÉIA - COL. MARISTA - Qtd Medições: 419
Praia: UBATUBA - TONINHAS - Qtd Medições: 429
Praia: SANTOS - GONZAGA - Qtd Medições: 429
Praia: ILHABELA - ARMAÇÃO - Qtd Medições: 419
Praia: ILHA COMPRIDA - PRAINHA (BALSA) - Qtd Medições: 102
Praia: ILHA COMPRIDA - PONTAL - Qtd Medições: 102
Praia: ILHA ANCHIETA - PRAINHA DO LESTE - Qtd Medições: 345
Praia: SANTOS - JOSÉ MENINO (R. OLAVO BILAC) - Qtd Medições: 419
Praia: ILHA ANCHIETA - PRAIA DO SUL - Qtd Medições: 346
Praia: UBATUBA - ENSEADA - Qtd Medições: 429
Praia: ILHA ANCHIETA - PRAIA DE FORA - Qtd Medições: 350
Praia: SÃO VICENTE - MILIONÁRIOS - Qtd Medições: 429
Praia: SANTOS - JOSÉ MENINO (R. FREDERICO OZANAN) - Qtd Medições: 429
Praia: ILHABELA - PINTO - Qtd Medições: 419
Praia: UBATUBA - MARANDUBA - Qtd Medições: 429
Praia: ILHA ANCHIETA - PRAIA DO PRESIDIO - Qtd Medições: 350
Praia: UBATUBA - SANTA RITA - Qtd Medições: 429
Praia: SÃO SEBASTIÃO - PRAINHA - Qtd Medições: 419
Praia: BERTIOGA - ENSEADA - INDAIÁ -

Praia: UBATUBA - IPEROIG - Qtd Medições: 419
Praia: ITANHAÉM - SUARÃO - Qtd Medições: 429
Praia: ITANHAÉM - CAMPOS ELÍSEOS - Qtd Medições: 418
Praia: ITANHAÉM - JARDIM SÃO FERNANDO - Qtd Medições: 419
Praia: PERUÍBE - PERUÍBE (R. ICARAÍBA) - Qtd Medições: 419
Praia: ITANHAÉM - JARDIM CIBRATEL - Qtd Medições: 429
Praia: PERUÍBE - PERUÍBE (PARQUE TURÍSTICO) - Qtd Medições: 429
Praia: PERUÍBE - PERUÍBE (BALN. SÃO JOÃO BATISTA) - Qtd Medições: 429
Praia: PRAIA GRANDE - GUILHERMINA - Qtd Medições: 429
Praia: UBATUBA - RIO  ITAMAMBUCA - Qtd Medições: 429
Praia: ILHABELA - JULIÃO - Qtd Medições: 412
Praia: ILHA COMPRIDA - BALNEÁRIO ADRIANA - Qtd Medições: 94
Praia: ILHABELA - BARREIROS NORTE - Qtd Medições: 366
Praia: ILHABELA - BARREIROS SUL - Qtd Medições: 366
Praia: ILHABELA - ENGENHO D'ÁGUA - Qtd Medições: 346
Praia: ITANHAÉM - JARDIM REGINA - Qtd Medições: 330
Praia: GUARUJÁ - IPORANGA - Qtd Medições: 75
Praia: MONGAGUÁ - FLÓRIDA MIRIM - Qtd Medições: 321
Praia: ITANHAÉM - SUARÃO - AFPES

In [31]:
df_aux=df.pivot(index='praia', columns='Date', values='Enterococcus' )
df_aux.fillna(df_aux.mean(), inplace=True)
df_aux=df_aux.astype('int32')
df_aux
#d = {'praia': [praia]}
#df_novo = pd.DataFrame(data=d) 
#pd.concat([df_novo, df_aux['Enterococcus']], axis=1)
#df_novo = df_novo.join(df_aux['Enterococcus'].T)
#df_novo

Date,2012-01-03,2012-01-08,2012-01-15,2012-01-22,2012-01-29,2012-02-05,2012-02-12,2012-02-19,2012-02-26,2012-03-04,...,2020-07-27,2020-08-03,2020-08-10,2020-08-17,2020-08-24,2020-08-31,2020-09-07,2020-09-14,2020-09-21,2020-09-28
praia,,,,,,,,,,,,,,,,,,,,,
BERTIOGA - BORACÉIA - COL. MARISTA,8,22,17,8,2,1,68,32,1,1,...,15,20,18,48,33,35,39,36,287,91
BERTIOGA - BORACÉIA - SUL,16,1,1,15,7,2,18,45,4,1,...,15,20,18,48,33,35,39,36,287,91
BERTIOGA - ENSEADA - COLÔNIA DO SESC,128,42,18,31,11,19,252,36,1,2,...,15,20,18,48,33,35,39,36,287,91
BERTIOGA - ENSEADA - INDAIÁ,17,85,1,10,108,6,7,29,15,1,...,15,20,18,48,33,35,39,36,287,91
BERTIOGA - ENSEADA - R. RAFAEL COSTABILI,204,34,19,33,10,25,288,57,42,2,...,3,43,3,30,30,8,2,2,152,61
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
UBATUBA - SUNUNGA,5,6,34,9,3,5,9,2,3,1,...,15,20,18,48,33,35,39,36,287,91
UBATUBA - TENÓRIO,9,96,42,14,7,17,5,1,12,5,...,9,1,3,14,2,1,12,1,86,4
UBATUBA - TONINHAS,1,128,21,11,13,2,13,21,28,2,...,36,9,3,7,9,18,10,2,188,41


In [42]:
test_size=20
treino = df_aux.iloc[:,:-test_size].copy()
teste = df_aux.iloc[:,-test_size:].copy()
print(treino.shape)
print(teste.shape)

(174, 409)
(174, 20)


In [44]:
gcn_lstm = GCN_LSTM(
    seq_len=20,
    adj=20,
    gc_layer_sizes=[16, 10],
    gc_activations=["relu", "relu"],
    lstm_layer_sizes=[200, 200],
    lstm_activations=["tanh", "tanh"],
)
x_input, x_output = gcn_lstm.in_out_tensors()
model = Model(inputs=x_input, outputs=x_output)
model.summary()

NameError: name 'seq_len' is not defined